# Brain Dance - Step 4: Integration Testing

This notebook tests the Mega-SAM → Instant4D pipeline on Google Colab.

**Prerequisites:**
- Google Colab Pro (T4/V100/A100 GPU)
- Your test video uploaded to Google Drive

**Estimated time:**
- First run: ~30 minutes (setup + compilation)
- Subsequent runs: ~15 minutes (pipeline only)

## Section A: Environment Setup

In [ ]:
# Cell 1: Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Clone repository with submodules
import os

if not os.path.exists('/content/brain-dance'):
    !git clone --recursive https://github.com/ujseah/brain-dance.git /content/brain-dance
else:
    print("Repository already cloned")
    
%cd /content/brain-dance

# Verify submodules are initialized
!ls -la instant4d/
!ls -la instant4d/SLAM/mega-sam/ 2>/dev/null || echo "Mega-SAM submodule not found - initializing..."
!git submodule update --init --recursive

In [ ]:
# Cell 3: Install base dependencies
print("Installing base dependencies...")
!pip install -q -r backend/requirements.txt

# Install GPU-specific packages
# Note: unidepth must be installed from GitHub, not PyPI
print("\nInstalling GPU packages...")
!pip install -q xformers plyfile gdown
!pip install -q git+https://github.com/lpiccinelli-eth/UniDepth.git

# Verify PyTorch CUDA
import torch
assert torch.cuda.is_available(), "CUDA not available after install!"
print(f"\nPyTorch CUDA: {torch.version.cuda}")

In [ ]:
# Cell 4: Compile Instant4D CUDA kernels (~10 minutes)
# This compiles: diff-gaussian-rasterization, pointops2, simple-knn, fused-ssim

print("Compiling Instant4D CUDA kernels...")
print("This takes ~10 minutes on first run.\n")

!bash scripts/setup_instant4d.sh

# Trigger JIT compilation in a fresh Python process to populate the cache
print("\nTriggering JIT compilation for gaussian_renderer...")
!rm -rf /root/.cache/torch_extensions/

# Write test script
with open('/tmp/jit_compile.py', 'w') as f:
    f.write("import sys\n")
    f.write("sys.path.insert(0, '/content/brain-dance/instant4d')\n")
    f.write("sys.path.insert(0, '/content/brain-dance/instant4d/submodule/fussed-ssim')\n")
    f.write("from gaussian_renderer import GaussianRasterizer\n")
    f.write("from fused_ssim import fused_ssim\n")
    f.write("print('[OK] JIT compilation complete')\n")

!python3 /tmp/jit_compile.py

In [ ]:
# Cell 5: Setup Mega-SAM (~10 minutes)
# This compiles lietorch, droid_backends and downloads checkpoints

print("Setting up Mega-SAM...")
print("This downloads ~1.5GB of checkpoints and compiles CUDA extensions.\n")

!bash scripts/setup_megasam.sh

In [ ]:
# Cell 6: Verify all installations
print("Verifying installations...\n")

import sys
sys.path.insert(0, '/content/brain-dance/instant4d')

# Instant4D - uses JIT compilation through gaussian_renderer
try:
    from gaussian_renderer import GaussianRasterizer, GaussianRasterizationSettings
    print("[OK] diff-gaussian-rasterization (via gaussian_renderer)")
except ImportError as e:
    print(f"[FAIL] diff-gaussian-rasterization: {e}")

try:
    from fused_ssim import fused_ssim
    print("[OK] fused-ssim")
except ImportError as e:
    print(f"[FAIL] fused-ssim: {e}")

# simple-knn and pointops are used internally by Instant4D
# We verify the compiled .so files exist instead of direct import
from pathlib import Path
instant4d_path = Path('/content/brain-dance/instant4d')

simple_knn_so = list((instant4d_path / 'submodule/simple-knn/simple_knn').glob('*.so'))
if simple_knn_so:
    print(f"[OK] simple-knn ({simple_knn_so[0].name})")
else:
    print("[FAIL] simple-knn: .so file not found")

pointops_so = list(instant4d_path.glob('submodule/pointops2/*.so'))
if pointops_so:
    print(f"[OK] pointops ({pointops_so[0].name})")
else:
    print("[FAIL] pointops: .so file not found")

# Mega-SAM kernels
sys.path.insert(0, '/content/brain-dance/instant4d/SLAM/mega-sam/base/thirdparty/lietorch')
try:
    import lietorch
    print("[OK] lietorch")
except ImportError as e:
    print(f"[FAIL] lietorch: {e}")

try:
    import droid_backends
    print("[OK] droid_backends")
except ImportError as e:
    print(f"[FAIL] droid_backends: {e}")

print("\nVerification complete!")

## Section B: Upload Your Video

Choose one of two options:
- **Option 1**: Direct upload (drag & drop) - simpler, no Drive needed
- **Option 2**: Google Drive mount - better for large files or re-runs

In [ ]:
# Cell 7a: OPTION 1 - Direct upload (drag & drop)
from google.colab import files
from pathlib import Path
from IPython.display import HTML, display
import shutil

print("Upload your video file (drag & drop or click to browse):")
uploaded = files.upload()

if uploaded:
    # Get the uploaded filename and move to /content/input.mp4
    uploaded_name = list(uploaded.keys())[0]
    LOCAL_VIDEO = "/content/input.mp4"
    shutil.move(uploaded_name, LOCAL_VIDEO)
    print(f"\nVideo saved to: {LOCAL_VIDEO}")
    
    # Show video info
    !ffprobe -v quiet -show_entries format=duration -show_entries stream=width,height,r_frame_rate -of csv=p=0 {LOCAL_VIDEO}
    
    # Preview video in notebook
    print("\nVideo preview:")
    display(HTML(f'''
    <video width="640" controls>
        <source src="/content/input.mp4" type="video/mp4">
    </video>
    '''))
else:
    print("No file uploaded. Run this cell again or use Option 2 (Drive mount).")

In [ ]:
# Cell 7b: OPTION 2 - Google Drive mount (for large files or re-runs)
# Skip this cell if you used Option 1 above

from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

# ============================================
# TODO: Update this path to your video file
# ============================================
VIDEO_PATH = "/content/drive/MyDrive/test_video.mp4"

# Copy to local storage for faster access
LOCAL_VIDEO = "/content/input.mp4"

if Path(VIDEO_PATH).exists():
    shutil.copy(VIDEO_PATH, LOCAL_VIDEO)
    print(f"Video copied to {LOCAL_VIDEO}")
    
    # Show video info
    !ffprobe -v quiet -show_entries format=duration -show_entries stream=width,height,r_frame_rate -of csv=p=0 {LOCAL_VIDEO}
else:
    print(f"ERROR: Video not found at {VIDEO_PATH}")
    print("Please update VIDEO_PATH to point to your video file.")

## Section C: Run Stage 1 (Mega-SAM Pose Estimation)

In [ ]:
# Cell 9: Process video with Mega-SAM
import sys
sys.path.insert(0, '/content/brain-dance/backend')

from stages.video_processing import VideoProcessingStage

# Configure Stage 1
config = {
    "pose_estimator": "megasam",  # Primary: Mega-SAM
    "max_frames": 150,             # Limit for testing (adjust based on video length)
    "megasam_opt_focal": True,     # Optimize focal length during SLAM
}

# Create stage and process video
stage1 = VideoProcessingStage(config)

print("Processing video with Mega-SAM...")
print("This includes: frame extraction, depth estimation, camera tracking\n")

stage1_result = stage1.process(
    video_path=LOCAL_VIDEO,
    output_dir="/content/stage1_output",
    progress_callback=lambda p, m: print(f"  [{p*100:.0f}%] {m}")
)

print(f"\n{'='*50}")
print(f"Stage 1 Complete!")
print(f"{'='*50}")
print(f"Frames extracted: {stage1_result.num_frames}")
print(f"Transforms: {stage1_result.transforms_path}")
print(f"Depth maps: {stage1_result.metadata.get('depth_maps_dir', 'N/A')}")
print(f"Motion prob: {stage1_result.metadata.get('motion_prob_path', 'N/A')}")

In [ ]:
# Cell 9b: Verify Stage 1 outputs
from pathlib import Path
import json

# Check transforms.json
transforms_path = Path(stage1_result.transforms_path)
if transforms_path.exists():
    with open(transforms_path) as f:
        transforms = json.load(f)
    print(f"Transforms: {len(transforms.get('frames', []))} frames")
    print(f"Camera model: {transforms.get('camera_model', 'N/A')}")
    if 'fl_x' in transforms:
        print(f"Focal length: {transforms['fl_x']:.1f}")

# Check depth maps (Mega-SAM specific)
depth_dir = stage1_result.metadata.get('depth_maps_dir')
if depth_dir and Path(depth_dir).exists():
    depth_files = list(Path(depth_dir).glob('*.npz'))
    print(f"\nDepth maps: {len(depth_files)} files")
else:
    print("\nNo Mega-SAM depth maps (using fallback path)")

# Check motion probability (Mega-SAM specific)
motion_path = stage1_result.metadata.get('motion_prob_path')
if motion_path and Path(motion_path).exists():
    import numpy as np
    motion_prob = np.load(motion_path)
    print(f"Motion probability: shape {motion_prob.shape}")
else:
    print("No Mega-SAM motion probability (using fallback)")

## Section D: Run Stage 3 (Instant4D 4D Training)

In [ ]:
# Cell 10: Train 4D Gaussians with Instant4D
from adapters.instant4d import Instant4DAdapter, Instant4DOptions

# Configure training options
options = Instant4DOptions(
    # Training parameters
    iterations=2000,        # Reduced for testing (default: 5000)
    batch_size=1,           # Images per batch
    
    # Mega-SAM integration
    use_megasam=True,       # Use depth/motion from Mega-SAM
    
    # Grid pruning
    enable_pruning=True,    # 92% Gaussian reduction
    
    # Export settings
    export_fps=10,          # 10 frames for quick test
)

# Create adapter and run pipeline
adapter = Instant4DAdapter()

print("Training 4D Gaussians with Instant4D...")
print(f"Iterations: {options.iterations}")
print(f"Export frames: {options.export_fps}\n")

stage3_result = adapter.run_full_pipeline(
    video_result=stage1_result,
    output_dir="/content/stage3_output",
    options=options,
    progress_callback=lambda p, m: print(f"  [{p*100:.0f}%] {m}")
)

print(f"\n{'='*50}")
print(f"Stage 3 Complete!")
print(f"{'='*50}")
print(f"Gaussians: {stage3_result.num_gaussians:,}")
print(f"PLY files: {len(stage3_result.ply_paths)}")
print(f"Model: {stage3_result.model_path}")
print(f"Metrics: {stage3_result.metrics}")

## Section E: Visualize Results

View extracted frames, depth maps, and Gaussian statistics directly in the notebook.

In [ ]:
# Cell 11: Visualize extracted frames and depth maps
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from PIL import Image

# Find frame images
frames_dir = Path("/content/stage1_output/images")
if not frames_dir.exists():
    frames_dir = Path("/content/stage1_output/frames")

if frames_dir.exists():
    frame_files = sorted(frames_dir.glob("*.png")) + sorted(frames_dir.glob("*.jpg"))
    
    # Show first, middle, and last frames
    if len(frame_files) >= 3:
        indices = [0, len(frame_files)//2, -1]
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for ax, idx in zip(axes, indices):
            img = Image.open(frame_files[idx])
            ax.imshow(img)
            ax.set_title(f"Frame {idx if idx >= 0 else len(frame_files)+idx}")
            ax.axis('off')
        plt.suptitle(f"Extracted Frames ({len(frame_files)} total)")
        plt.tight_layout()
        plt.show()
else:
    print("No frames directory found")

# Show depth maps if available
depth_dir = Path("/content/stage1_output/depth")
if depth_dir.exists():
    depth_files = sorted(depth_dir.glob("*.npz")) + sorted(depth_dir.glob("*.npy"))
    if depth_files:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        indices = [0, len(depth_files)//2, -1]
        for ax, idx in zip(axes, indices):
            if depth_files[idx].suffix == '.npz':
                depth = np.load(depth_files[idx])['depth']
            else:
                depth = np.load(depth_files[idx])
            ax.imshow(depth, cmap='viridis')
            ax.set_title(f"Depth {idx if idx >= 0 else len(depth_files)+idx}")
            ax.axis('off')
        plt.suptitle(f"Depth Maps ({len(depth_files)} total)")
        plt.tight_layout()
        plt.show()
else:
    print("No depth maps found (may not be generated yet)")

In [ ]:
# Cell 12: Visualize Gaussian statistics
from plyfile import PlyData
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

ply_dir = Path("/content/stage3_output/plys")
if ply_dir.exists():
    ply_files = sorted(ply_dir.glob("*.ply"))
    print(f"Found {len(ply_files)} PLY files\n")
    
    # Collect Gaussian counts per frame
    counts = []
    for ply_path in ply_files:
        ply = PlyData.read(str(ply_path))
        counts.append(len(ply['vertex']))
    
    # Plot Gaussian count over time
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Line plot
    axes[0].plot(counts, 'b-', linewidth=2)
    axes[0].fill_between(range(len(counts)), counts, alpha=0.3)
    axes[0].set_xlabel('Frame')
    axes[0].set_ylabel('Number of Gaussians')
    axes[0].set_title('Gaussians per Frame')
    axes[0].grid(True, alpha=0.3)
    
    # Histogram
    axes[1].hist(counts, bins=20, edgecolor='black', alpha=0.7)
    axes[1].axvline(np.mean(counts), color='r', linestyle='--', label=f'Mean: {np.mean(counts):,.0f}')
    axes[1].set_xlabel('Number of Gaussians')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of Gaussian Counts')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"Gaussian count statistics:")
    print(f"  Min: {min(counts):,}")
    print(f"  Max: {max(counts):,}")
    print(f"  Mean: {np.mean(counts):,.0f}")
    print(f"  Std: {np.std(counts):,.0f}")
else:
    print("PLY directory not found! Run Stage 3 first.")

In [ ]:
# Cell 13: Check temporal metadata
if stage3_result.temporal_metadata:
    meta = stage3_result.temporal_metadata
    print("Temporal metadata:")
    print(f"  FPS: {meta.get('fps', 'N/A')}")
    print(f"  Duration: {meta.get('duration_seconds', 'N/A')} seconds")
    print(f"  Num frames: {meta.get('num_frames', 'N/A')}")
    
    timestamps = meta.get('timestamps', [])
    if timestamps:
        print(f"  Timestamps: {timestamps[:3]}...{timestamps[-3:]}")
else:
    print("No temporal metadata available")

## Section F: Download Results

Download the output files directly from the notebook (no Drive needed).

In [ ]:
# Cell 14: Package and download results
import shutil
from pathlib import Path
from datetime import datetime
from google.colab import files
import zipfile
import os

# Create timestamped output directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path(f"/content/brain_dance_output_{timestamp}")
output_dir.mkdir(parents=True, exist_ok=True)

# Copy Stage 1 outputs
print("Packaging Stage 1 outputs...")
if Path("/content/stage1_output").exists():
    shutil.copytree("/content/stage1_output", output_dir / "stage1", dirs_exist_ok=True)

# Copy Stage 3 outputs  
print("Packaging Stage 3 outputs...")
if Path("/content/stage3_output").exists():
    shutil.copytree("/content/stage3_output", output_dir / "stage3", dirs_exist_ok=True)

# Create zip file
zip_path = f"/content/brain_dance_results_{timestamp}.zip"
print(f"\nCreating zip archive: {zip_path}")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(output_dir):
        for file in files_list:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, output_dir)
            zipf.write(file_path, arcname)

# Show zip contents summary
zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"Zip size: {zip_size_mb:.1f} MB")

# Count files
stage1_files = len(list((output_dir / "stage1").rglob("*"))) if (output_dir / "stage1").exists() else 0
stage3_files = len(list((output_dir / "stage3").rglob("*"))) if (output_dir / "stage3").exists() else 0
print(f"Stage 1 files: {stage1_files}")
print(f"Stage 3 files: {stage3_files}")

# Trigger download
print("\nStarting download...")
files.download(zip_path)

print("\nDownload started! Check your browser's download folder.")

## Section G: Run Integration Tests (Optional)

In [ ]:
# Cell 15: Run pytest integration tests
%cd /content/brain-dance

print("Running setup verification tests...")
!pytest tests/test_instant4d_setup.py -v -m gpu --tb=short

print("\nRunning Instant4D adapter tests...")
!pytest tests/adapters/test_instant4d_adapter.py -v -m gpu --tb=short

## Summary

If all cells ran successfully, Step 4 integration testing is complete!

**What was tested:**
1. CUDA kernel compilation (Instant4D + Mega-SAM)
2. Stage 1: Mega-SAM pose estimation → depth maps + motion probability
3. Stage 3: Instant4D 4D training → per-frame PLY export

**Next steps:**
- Stage 5: Web export (PLY → SPZ compression)
- Frontend viewer with temporal playback